In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DataType,BooleanType
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catelog_name","ecommers","Catalog Name")
dbutils.widgets.text("storage_account_name",'ecommcistorage05',"storage account name")
dbutils.widgets.text("container","ecom-raw-data","Container Name")


In [0]:
catelog_name=dbutils.widgets.get("catelog_name")
storage_account_name=dbutils.widgets.get("storage_account_name")
container=dbutils.widgets.get("container")

In [0]:
silver_checkpoint_path = (
    f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/"
    "checkpoint/silver/fact_order_items/"
)

df = spark.readStream \
    .format("delta") \
    .table(f"{catelog_name}.bronze.brz_order_items")

df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", silver_checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable(f"{catelog_name}.silver.fact_order_items") \
    .awaitTermination()

In [0]:
# df=spark.table(f"{catelog_name}.silver.fact_order_items")

In [0]:
df = df.dropDuplicates(["order_id", "item_seq"])

# Transformation: Convert 'Two' → 2 and cast to Integer
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

# Transformation : Remove any '$' or other symbols from unit_price, keep only numeric
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

# Transformation : Remove '%' from discount_pct and cast to double
df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

# Transformation : coupon code processing (convert to lower)
df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

# Transformation : channel processing 
df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

#Transformation : Add processed time 
df = df.withColumn(
    "processed_time", F.current_timestamp()
)


In [0]:
silver_checkpoint_path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items/"
print(silver_checkpoint_path)

In [0]:
from delta.tables import DeltaTable

def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catelog_name}.silver.slv_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

slv_checkpoint_path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/slv_order_items/"

df.writeStream.foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", slv_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    availableNow=True
).start().awaitTermination()